# Analysing Complex Crystal Structures: Argyrodite Solid Electrolytes

This tutorial demonstrates how to apply `site-analysis` to a realistic materials science problem: analysing lithium-ion site occupations in Li6PS5Cl argyrodite solid electrolytes. You will learn how to define multiple crystallographically distinct site types using a reference structure with dummy atoms, handle structure alignment and site mapping between structures with different chemical compositions, and examine how site occupations change with varying degrees of S/Cl anion disorder.

## Prerequisites

This tutorial requires the following packages:

- `site-analysis`

This tutorial uses trajectory data from molecular dynamics simulations.
The data files are included in the `site-analysis` GitHub repository at
`tutorials/data/`:

- `Li6PS5Cl_0p_XDATCAR.gz` — 0% anion site exchange (fully ordered)
- `Li6PS5Cl_50p_XDATCAR.gz` — 50% anion site exchange
- `Li6PS5Cl_100p_XDATCAR.gz` — 100% anion site exchange

These files are available in the [tutorials/data](https://github.com/bjmorgan/site-analysis/tree/main/tutorials/data) directory of the GitHub repository. The code examples below assume you are running from the repository root directory.

## Background: the argyrodite structure

The argyrodite Li6PS5Cl is a tetrahedrally close-packed structure where the anions (S2-, Cl-) form the vertices of tetrahedra. The centres of these tetrahedra are interstitial sites that can be classified into six crystallographically distinct types, numbered 0 to 5:

- **Type 0** sites are occupied by phosphorus (forming PS4 tetrahedra)
- **Types 1–5** are, in principle, available for lithium occupation

In Li6PS5Cl, the S and Cl anions can exchange between the 4a and 4c Wyckoff positions. This anion disorder modifies the local coordination environments around lithium sites, affecting which sites lithium ions prefer to occupy. Increased disorder activates previously unfavourable site types (particularly type 4), leading to enhanced ionic conductivity.

For more details, see [Morgan (2021)](https://doi.org/10.1021/acs.chemmater.0c03738).

In this tutorial we analyse three MD simulations with different degrees of S/Cl site exchange (0%, 50%, and 100%) and calculate the time-averaged distribution of lithium ions over the five interstitial site types.

## Creating a reference structure

To define multiple site types with polyhedral sites, we need a reference structure where each site type is marked by a different dummy atom species. This lets us call `with_polyhedral_sites` separately for each type. For more details on the reference structure workflow, see the [reference workflow guide](https://site-analysis.readthedocs.io/en/latest/guides/reference_workflow.html).

In [ ]:
%config InlineBackend.figure_format = 'retina'

In [ ]:
from pymatgen.io.vasp import Xdatcar
from pymatgen.core import Structure, Lattice
import numpy as np
from collections import Counter
from site_analysis import TrajectoryBuilder

lattice = Lattice.cubic(a=10.155)

coords = np.array(
    [[0.5,     0.5,     0.5],     # P (type 0) - PS4 tetrahedra
     [0.9,     0.9,     0.6],     # type 1 (Li in reference)
     [0.23,    0.92,    0.09],    # type 2 (Mg in reference)
     [0.25,    0.25,    0.25],    # type 3 (Na in reference)
     [0.15,    0.15,    0.15],    # type 4 (Be in reference)
     [0.0,     0.183,   0.183],   # type 5 (K in reference)
     [0.0,     0.0,     0.0],     # S - anion position (4a site)
     [0.75,    0.25,    0.25],    # S - anion position (4c site)
     [0.11824, 0.11824, 0.38176]] # S - anion position (16e site)
)

reference_structure = Structure.from_spacegroup(
    sg="F-43m",
    lattice=lattice,
    species=['P', 'Li', 'Mg', 'Na', 'Be', 'K', 'S', 'S', 'S'],
    coords=coords) * [2, 2, 2]

print(f"Reference structure contains {len(reference_structure)} atoms")
print(f"Composition: {reference_structure.composition.formula}")

Each dummy species (Li, Mg, Na, Be, K) marks a different tetrahedral site type. The reference uses all-S anions — we will handle the S/Cl disorder through species mapping later.

## Building the trajectory analyser

We now write a function that configures a `TrajectoryBuilder` for the argyrodite analysis (see the [builders guide](https://site-analysis.readthedocs.io/en/latest/guides/builders.html) for a full overview). This demonstrates several advanced features:

In [ ]:
def build_trajectory(structure):
    builder = TrajectoryBuilder()

    # Set the reference and target structures
    builder.with_reference_structure(reference_structure)
    builder.with_structure(structure)

    # Specify lithium as the mobile species
    builder.with_mobile_species('Li')

    # Define five tetrahedral site types, one per dummy species
    builder.with_polyhedral_sites(
        centre_species='Li', vertex_species='S',
        cutoff=3.0, n_vertices=4, label='type 1')

    builder.with_polyhedral_sites(
        centre_species='Mg', vertex_species='S',
        cutoff=3.0, n_vertices=4, label='type 2')

    builder.with_polyhedral_sites(
        centre_species='Na', vertex_species='S',
        cutoff=3.0, n_vertices=4, label='type 3')

    builder.with_polyhedral_sites(
        centre_species='Be', vertex_species='S',
        cutoff=3.0, n_vertices=4, label='type 4')

    builder.with_polyhedral_sites(
        centre_species='K', vertex_species='S',
        cutoff=3.0, n_vertices=4, label='type 5')

    # Align the reference and target structures using phosphorus positions
    builder.with_structure_alignment(align_species='P')

    # Map between S and Cl -- the reference has all-S anions, but the
    # real structures have a mix of S and Cl at these positions
    builder.with_site_mapping(mapping_species=['S', 'Cl'])

    trajectory = builder.build()
    return trajectory

Key points:

- We call `with_polyhedral_sites` five times, once per site type, using a different `centre_species` each time. For more details on polyhedral sites, see the [polyhedral sites guide](https://site-analysis.readthedocs.io/en/latest/guides/polyhedral_sites.html).
- `with_structure_alignment(align_species='P')` aligns the reference to the target structure using the phosphorus atom positions.
- `with_site_mapping(mapping_species=['S', 'Cl'])` tells the builder that S and Cl are interchangeable when mapping between the all-S reference and the real (disordered) structure.

## Analysing site occupations

We define a helper function to calculate the percentage of time lithium ions spend in each site type:

In [ ]:
def print_site_occupations(trajectory, title=None):
    site_types = ['type 5', 'type 4', 'type 3', 'type 2', 'type 1']

    site_labels = []
    for atom in trajectory.atoms:
        for site_idx in atom.trajectory:
            if site_idx is not None:
                site_labels.append(trajectory.sites[site_idx].label)

    c = Counter(site_labels)
    total_sites = sum(c.values())

    if title:
        print(f"\nSite occupation analysis - {title}:")
        print("-" * 40)

    for t in site_types:
        percentage = (c.get(t, 0) / total_sites * 100) if total_sites > 0 else 0
        print(f'{t}: {percentage:.2f}%')

## Ordered structure (0% anion site exchange)

We start with the fully ordered Li6PS5Cl, where S and Cl occupy their equilibrium Wyckoff positions with no site exchange:

In [ ]:
md_structures = Xdatcar('data/Li6PS5Cl_0p_XDATCAR.gz').structures
print(f"Loaded trajectory with {len(md_structures)} frames")

trajectory_0p = build_trajectory(md_structures[0])
trajectory_0p.trajectory_from_structures(md_structures, progress=True)

print_site_occupations(trajectory_0p, "0% anion disorder")

In the ordered structure, lithium ions predominantly occupy type 5 sites (~80%) with type 2 as the secondary preference (~20%). Other site types are essentially unoccupied.

## Partially disordered structure (50% anion site exchange)

With 50% S/Cl site exchange — maximal anion disorder:

In [ ]:
md_structures = Xdatcar('data/Li6PS5Cl_50p_XDATCAR.gz').structures
print(f"Loaded trajectory with {len(md_structures)} frames")

trajectory_50p = build_trajectory(md_structures[0])
trajectory_50p.trajectory_from_structures(md_structures, progress=True)

print_site_occupations(trajectory_50p, "50% anion disorder")

With 50% disorder, type 5 occupation decreases while type 2 and type 4 occupation increases.

## Fully inverted structure (100% anion site exchange)

With complete S/Cl site exchange:

In [ ]:
md_structures = Xdatcar('data/Li6PS5Cl_100p_XDATCAR.gz').structures
print(f"Loaded trajectory with {len(md_structures)} frames")

trajectory_100p = build_trajectory(md_structures[0])
trajectory_100p.trajectory_from_structures(md_structures, progress=True)

print_site_occupations(trajectory_100p, "100% anion disorder")

## Comparing results across disorder levels

The trend across the three simulations is clear:

| Site type | 0% disorder | 50% disorder | 100% disorder |
|-----------|-------------|--------------|---------------|
| Type 5    | 80.20%      | 65.92%       | 53.39%        |
| Type 4    | 0.02%       | 2.63%        | 7.33%         |
| Type 3    | 0.00%       | 0.00%        | 0.00%         |
| Type 2    | 19.78%      | 31.43%       | 39.28%        |
| Type 1    | 0.01%       | 0.02%        | 0.00%         |

With increasing anion disorder:

- Type 5 occupation **decreases** from ~80% to ~53%
- Type 2 occupation **increases** from ~20% to ~39%
- Type 4 occupation **emerges**, growing from negligible to ~7%
- Type 3 remains unoccupied regardless of disorder level

The activation of type 4 sites with increasing disorder is significant — these sites provide additional pathways for lithium diffusion, contributing to the enhanced ionic conductivity observed experimentally in disordered argyrodites.

## Summary

This tutorial demonstrated several advanced `site_analysis` features for complex crystal structures:

- **Multiple site types**: calling `with_polyhedral_sites` multiple times with different `centre_species` to define crystallographically distinct sites
- **Structure alignment**: using `with_structure_alignment` to align reference and target structures via a shared atomic species
- **Species mapping**: using `with_site_mapping` with multiple species to handle structures where two species (S and Cl) are interchangeable between the reference and real structures

These techniques are applicable to any material with multiple distinct site types or chemical disorder between structurally equivalent positions.